In [29]:
import json
import sys
sys.path.insert(0, 'base/src')
import spacy
from spacy.training.example import Example
from spacy.scorer import Scorer

from huggingface_hub import snapshot_download
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline


In [30]:
def unified_to_spacy(data):
    """Convert unified format to spaCy format."""
    return [
        data["text"],
        {"entities": [[e["start"], e["end"], e["label"]] for e in data["entities"]]}
    ]

def test_model_spacy(model, test_data):
    print(f"Test samples: {len(test_data)}\n")

    examples = []
    for text, annotations in test_data:
        doc = model.make_doc(text)
        example = Example.from_dict(doc, annotations)
        example.predicted = model(text)
        examples.append(example)

    scorer = Scorer()
    scores = scorer.score(examples)

    print(f"Precision: {scores['ents_p']:.2%}")
    print(f"Recall: {scores['ents_r']:.2%}")
    print(f"F1-Score: {scores['ents_f']:.2%}")
    print(f"\nPer-entity scores:")
    for entity_type, metrics in scores['ents_per_type'].items():
        print(f"  {entity_type}:")
        print(f"    Precision: {metrics['p']:.2%}")
        print(f"    Recall: {metrics['r']:.2%}")
        print(f"    F1-Score: {metrics['f']:.2%}")



In [31]:
with open('../../data/processed/travel-order-dataset-test.json', 'r') as f:
    test_data = json.load(f)

test_data_spacy = [unified_to_spacy(item) for item in test_data]

test_sentences = [
    "Je veux aller de paris a marseille",
    "Un billet de Lyon pour Bordeaux s'il vous plait",
    "Je dois partir de Toulouse vers Nice demain",
    "Trajet entre Nantes et Strasbourg lundi prochain",
    "Je cherche un train de Montpellier a Lille a 14h",
    "Horaires des TGV de Rennes a La Rochelle ce vendredi",
    "Je pars de Clermont-Ferrand direction Aix-en-Provence dans l'apres-midi",
    "Y a-t-il un train de Grenoble a Geneve ce soir ?",
]

## Test Spacy models

In [32]:
print(f"Test samples: {len(test_data)}")

model_path = snapshot_download(repo_id="YanisC/fr_travel_order_ner_model")
hf_spacy_model = spacy.load(model_path)


print("\n=== Evaluating on Test Data ===")
test_model_spacy(hf_spacy_model, test_data_spacy)

Invalid model-index. Not loading eval results into CardData.


Test samples: 2000


Fetching 25 files: 100%|██████████| 25/25 [00:00<00:00, 90785.80it/s]



=== Evaluating on Test Data ===
Test samples: 2000

Precision: 96.27%
Recall: 96.18%
F1-Score: 96.22%

Per-entity scores:
  TIME:
    Precision: 95.29%
    Recall: 96.38%
    F1-Score: 95.83%
  DEPARTURE:
    Precision: 96.15%
    Recall: 96.30%
    F1-Score: 96.23%
  DESTINATION:
    Precision: 97.00%
    Recall: 95.92%
    F1-Score: 96.46%


In [33]:
print("=== Testing spacy model on Real Queries ===\n")
for query in test_sentences:
    print(f"Query: {query}")
    doc = hf_spacy_model(query)
    entities = [(ent.text, ent.label_) for ent in doc.ents]
    print(f"SPACY model found: {entities}")

=== Testing spacy model on Real Queries ===

Query: Je veux aller de paris a marseille
SPACY model found: [('paris a', 'DEPARTURE'), ('marseille', 'DESTINATION')]
Query: Un billet de Lyon pour Bordeaux s'il vous plait
SPACY model found: []
Query: Je dois partir de Toulouse vers Nice demain
SPACY model found: [('Toulouse', 'DEPARTURE'), ('demain', 'TIME')]
Query: Trajet entre Nantes et Strasbourg lundi prochain
SPACY model found: [('lundi prochain', 'TIME')]
Query: Je cherche un train de Montpellier a Lille a 14h
SPACY model found: [('a 14h', 'TIME')]
Query: Horaires des TGV de Rennes a La Rochelle ce vendredi
SPACY model found: [('Rennes', 'DEPARTURE'), ('ce vendredi', 'TIME')]
Query: Je pars de Clermont-Ferrand direction Aix-en-Provence dans l'apres-midi
SPACY model found: [('direction', 'DESTINATION'), ("dans l'apres-midi", 'TIME')]
Query: Y a-t-il un train de Grenoble a Geneve ce soir ?
SPACY model found: [('ce soir', 'TIME')]


## Test camemBERT Models

In [35]:
HF_REPO = "YanisC/camembert-ner-travel"

bert_tokenizer = AutoTokenizer.from_pretrained(HF_REPO)
bert_model = AutoModelForTokenClassification.from_pretrained(HF_REPO)

camembert_pipe = pipeline("ner", model=bert_model, tokenizer=bert_tokenizer, aggregation_strategy="simple")

print("\n=== Testing CamemBERT model on Real Queries ===\n")
for query in test_sentences:
    results = camembert_pipe(query)
    entities = {"DEPARTURE": [], "DESTINATION": [], "TIME": []}
    for ent in results:
        if ent["entity_group"] in entities:
            entities[ent["entity_group"]].append(ent["word"])

    print(f"Query:        {query}")
    print(f"  Departure : {entities['DEPARTURE'][0] if entities['DEPARTURE'] else 'Not found'} | Destination : {entities['DESTINATION'][0] if entities['DESTINATION'] else 'Not found'} | Time : {entities['TIME'][0] if entities['TIME'] else 'Not found'}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 499.06it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              



=== Testing CamemBERT model on Real Queries ===

Query:        Je veux aller de paris a marseille
  Departure : paris | Destination : marseille | Time : Not found
Query:        Un billet de Lyon pour Bordeaux s'il vous plait
  Departure : Lyon | Destination : Bordeaux | Time : Not found
Query:        Je dois partir de Toulouse vers Nice demain
  Departure : Toulouse | Destination : Nice | Time : demain
Query:        Trajet entre Nantes et Strasbourg lundi prochain
  Departure : Nantes | Destination : Strasbourg | Time : lundi prochain
Query:        Je cherche un train de Montpellier a Lille a 14h
  Departure : Montpellier | Destination : Lille | Time : a 14h
Query:        Horaires des TGV de Rennes a La Rochelle ce vendredi
  Departure : Rennes | Destination : La Rochelle | Time : ce vendredi
Query:        Je pars de Clermont-Ferrand direction Aix-en-Provence dans l'apres-midi
  Departure : Clermont-Ferrand | Destination : Aix-en-Provence | Time : dans l'apres-midi
Query:        Y a-t